# MedTrack_DV — 02. Data Cleaning

Cleans all 3 datasets independently at the per-table level. This notebook does **not** build the four
final analytical tables (that is `03_data_normalization.ipynb`) — it stops once every source table is
cleaned, type-corrected, and saved to `data/processed/`.

**Dataset roles** (read this before touching any code below):

1. **HMIS (19 tables) — BACKBONE.** Profiling found zero orphaned foreign keys, zero duplicate primary
   keys, zero duplicate rows, and zero bad admission/discharge date ordering across every table. Cleaning
   here is type/format standardization, not error correction.
2. **Beds Management (4 tables) — SUPPLEMENT.** No shared key with HMIS (`patient_id`/`staff_id` are
   unrelated ID spaces). Cleaning here is per-table only; the department/week bridge to HMIS happens in
   `03_data_normalization`, not here.
3. **Readmission dataset (1 table) — BENCHMARK ONLY.** Operational fields (dates, bed counts,
   hospital/doctor IDs) fail internal logic checks on ~28-47% of rows and are **flagged, not fixed**.
   Only `discharge_status` / `readmission` / `patient_disease` are trustworthy enough to use downstream.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Every path below is defined ONCE and never reassigned - this avoids a
# fragile pattern from an earlier draft where a single PROCESSED_DIR
# variable was repeatedly repointed at different folders across cells,
# which broke as soon as cells were re-run out of order.
HMIS_RAW_DIR         = PROJECT_ROOT / "data" / "raw" / "Hospital HMIS Dataset for Healthcare Analytics" / "hospital_synthetic_shalaka" / "hospital_data"
BEDS_RAW_DIR         = PROJECT_ROOT / "data" / "raw" / "Hospital Beds Management"
READMISSION_RAW_DIR  = PROJECT_ROOT / "data" / "raw" / "Hospital Data for Patient Readmission Prediction"

HMIS_PROCESSED_DIR         = PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics"
BEDS_PROCESSED_DIR         = PROCESSED_DIR / "Hospital Beds Management"
READMISSION_PROCESSED_DIR  = PROCESSED_DIR / "Hospital Data for Patient Readmission Prediction"

for d in [HMIS_PROCESSED_DIR, BEDS_PROCESSED_DIR, READMISSION_PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [2]:
# =========================================================
# HELPER FUNCTIONS - defined ONCE, reused for all 3 datasets
# =========================================================

def strip_string_columns(df):
    """Remove leading/trailing spaces from every text column.
    Doesn't change values, just removes invisible formatting issues
    that can break joins/grouping later (e.g. 'Male ' != 'Male')."""
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
    return df

def convert_to_datetime(df, date_cols):
    """Convert text date columns to real datetime type.
    errors='coerce' turns any unparseable date into NaT instead of
    crashing, so we can spot bad dates instead of losing the row."""
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def report_missing(df, table_name):
    """Print % missing per column, only if any exist."""
    pct = (df.isnull().sum() / len(df) * 100).round(2)
    pct = pct[pct > 0]
    if len(pct) > 0:
        print(f"[{table_name}] missing values:\n{pct}\n")
    else:
        print(f"[{table_name}] no missing values")

def report_duplicates(df, table_name):
    """Print count of fully duplicated rows."""
    print(f"[{table_name}] duplicate rows: {df.duplicated().sum()}")

def check_foreign_key(child_df, child_col, parent_df, parent_col, child_name, parent_name):
    """Check every value in child_col exists in the parent table.
    We only REPORT orphans here - we never silently drop rows,
    since that would break the joins other tables depend on."""
    orphans = ~child_df[child_col].isin(parent_df[parent_col])
    print(f"[{child_name}.{child_col} -> {parent_name}.{parent_col}] orphan rows: {orphans.sum()}")
    return orphans

## Part 1 — HMIS Cleaning (19 tables)

In [3]:
hmis_file_names = [
    "department.csv", "patient.csv", "employee.csv", "disease.csv",
    "insurance_provider.csv", "drug_manufacturer.csv",
    "doctor.csv", "ward.csv", "drug.csv", "patient_insurance.csv",
    "bed.csv", "drug_inventory.csv",
    "staff_assignment.csv", "admission.csv",
    "diagnostic_test.csv", "patient_diagnostic.csv", "prescription.csv",
    "billing.csv", "billing_detail.csv",
]
dataframes = {f[:-4]: pd.read_csv(HMIS_RAW_DIR / f) for f in hmis_file_names}
cleaned_dataframes = {k: v.copy(deep=True) for k, v in dataframes.items()}
print(f"Loaded {len(dataframes)} HMIS raw tables")

Loaded 19 HMIS raw tables


### Parent / Master Tables

In [4]:
# --- department ---
df = cleaned_dataframes['department']
df = strip_string_columns(df)
report_missing(df, 'department'); report_duplicates(df, 'department')
cleaned_dataframes['department'] = df

# --- patient ---
df = cleaned_dataframes['patient']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['date_of_birth'])
# contact_number comes in many formats (dashes, dots, brackets, extensions
# like 'x90630'). Keep the ORIGINAL column untouched, add a clean
# digits-only version alongside it for anything needing a consistent format.
df['contact_number_clean'] = (
    df['contact_number'].str.split('x').str[0]
    .str.replace(r'\D', '', regex=True).str[-10:]
)
report_missing(df, 'patient'); report_duplicates(df, 'patient')
cleaned_dataframes['patient'] = df

# --- employee ---
df = cleaned_dataframes['employee']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['date_of_joining'])
report_missing(df, 'employee'); report_duplicates(df, 'employee')
cleaned_dataframes['employee'] = df

# --- disease ---
df = cleaned_dataframes['disease']
df = strip_string_columns(df)
report_missing(df, 'disease'); report_duplicates(df, 'disease')
cleaned_dataframes['disease'] = df

# --- insurance_provider ---
df = cleaned_dataframes['insurance_provider']
df = strip_string_columns(df)
report_missing(df, 'insurance_provider'); report_duplicates(df, 'insurance_provider')
cleaned_dataframes['insurance_provider'] = df

# --- drug_manufacturer ---
df = cleaned_dataframes['drug_manufacturer']
df = strip_string_columns(df)
report_missing(df, 'drug_manufacturer'); report_duplicates(df, 'drug_manufacturer')
cleaned_dataframes['drug_manufacturer'] = df

[department] no missing values
[department] duplicate rows: 0
[patient] no missing values
[patient] duplicate rows: 0
[employee] no missing values
[employee] duplicate rows: 0
[disease] no missing values
[disease] duplicate rows: 0
[insurance_provider] no missing values
[insurance_provider] duplicate rows: 0
[drug_manufacturer] no missing values
[drug_manufacturer] duplicate rows: 0


### First-Level Dependent Tables

In [5]:
# --- doctor ---
df = cleaned_dataframes['doctor']
df = strip_string_columns(df)
check_foreign_key(df, 'employee_id', cleaned_dataframes['employee'], 'employee_id', 'doctor', 'employee')
report_missing(df, 'doctor'); report_duplicates(df, 'doctor')
cleaned_dataframes['doctor'] = df

# --- ward ---
df = cleaned_dataframes['ward']
df = strip_string_columns(df)
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'ward', 'department')
report_missing(df, 'ward'); report_duplicates(df, 'ward')
cleaned_dataframes['ward'] = df

# --- drug ---
df = cleaned_dataframes['drug']
df = strip_string_columns(df)
check_foreign_key(df, 'manufacturer_id', cleaned_dataframes['drug_manufacturer'], 'manufacturer_id', 'drug', 'drug_manufacturer')
report_missing(df, 'drug'); report_duplicates(df, 'drug')
cleaned_dataframes['drug'] = df

# --- patient_insurance ---
df = cleaned_dataframes['patient_insurance']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['policy_start_date', 'policy_end_date'])
check_foreign_key(df, 'patient_id', cleaned_dataframes['patient'], 'patient_id', 'patient_insurance', 'patient')
check_foreign_key(df, 'insurance_provider_id', cleaned_dataframes['insurance_provider'], 'insurance_provider_id', 'patient_insurance', 'insurance_provider')
report_missing(df, 'patient_insurance'); report_duplicates(df, 'patient_insurance')
cleaned_dataframes['patient_insurance'] = df

[doctor.employee_id -> employee.employee_id] orphan rows: 0
[doctor] no missing values
[doctor] duplicate rows: 0
[ward.department_id -> department.department_id] orphan rows: 0
[ward] no missing values
[ward] duplicate rows: 0
[drug.manufacturer_id -> drug_manufacturer.manufacturer_id] orphan rows: 0
[drug] no missing values
[drug] duplicate rows: 0
[patient_insurance.patient_id -> patient.patient_id] orphan rows: 0
[patient_insurance.insurance_provider_id -> insurance_provider.insurance_provider_id] orphan rows: 0
[patient_insurance] no missing values
[patient_insurance] duplicate rows: 0


### Operational Resources

In [6]:
# --- bed ---
df = cleaned_dataframes['bed']
df = strip_string_columns(df)
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'bed', 'ward')
report_missing(df, 'bed'); report_duplicates(df, 'bed')
cleaned_dataframes['bed'] = df

# --- drug_inventory ---
df = cleaned_dataframes['drug_inventory']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['last_restock_date'])
check_foreign_key(df, 'drug_id', cleaned_dataframes['drug'], 'drug_id', 'drug_inventory', 'drug')
report_missing(df, 'drug_inventory'); report_duplicates(df, 'drug_inventory')
cleaned_dataframes['drug_inventory'] = df

[bed.ward_id -> ward.ward_id] orphan rows: 0
[bed] no missing values
[bed] duplicate rows: 0
[drug_inventory.drug_id -> drug.drug_id] orphan rows: 0
[drug_inventory] no missing values
[drug_inventory] duplicate rows: 0


### Hospital Transactions (admission is the spine table — extra care here)

In [7]:
# --- staff_assignment ---
df = cleaned_dataframes['staff_assignment']
df = strip_string_columns(df)
check_foreign_key(df, 'employee_id', cleaned_dataframes['employee'], 'employee_id', 'staff_assignment', 'employee')
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'staff_assignment', 'ward')
report_missing(df, 'staff_assignment'); report_duplicates(df, 'staff_assignment')
cleaned_dataframes['staff_assignment'] = df

# --- admission ---
df = cleaned_dataframes['admission']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['admission_date', 'discharge_date'])

# Sanity check: discharge should never be before admission.
bad_dates = df[df['discharge_date'] < df['admission_date']]
print(f"[admission] rows with discharge before admission: {len(bad_dates)}")

check_foreign_key(df, 'patient_id', cleaned_dataframes['patient'], 'patient_id', 'admission', 'patient')
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'admission', 'department')
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'admission', 'ward')
check_foreign_key(df, 'bed_id', cleaned_dataframes['bed'], 'bed_id', 'admission', 'bed')
check_foreign_key(df, 'disease_id', cleaned_dataframes['disease'], 'disease_id', 'admission', 'disease')

report_missing(df, 'admission'); report_duplicates(df, 'admission')
cleaned_dataframes['admission'] = df

[staff_assignment.employee_id -> employee.employee_id] orphan rows: 0
[staff_assignment.ward_id -> ward.ward_id] orphan rows: 0
[staff_assignment] no missing values
[staff_assignment] duplicate rows: 0
[admission] rows with discharge before admission: 0
[admission.patient_id -> patient.patient_id] orphan rows: 0
[admission.department_id -> department.department_id] orphan rows: 0
[admission.ward_id -> ward.ward_id] orphan rows: 0
[admission.bed_id -> bed.bed_id] orphan rows: 0
[admission.disease_id -> disease.disease_id] orphan rows: 0
[admission] no missing values
[admission] duplicate rows: 0


### Clinical Transactions

In [8]:
# --- diagnostic_test ---
df = cleaned_dataframes['diagnostic_test']
df = strip_string_columns(df)
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'diagnostic_test', 'department')
report_missing(df, 'diagnostic_test'); report_duplicates(df, 'diagnostic_test')
cleaned_dataframes['diagnostic_test'] = df

# --- patient_diagnostic ---
df = cleaned_dataframes['patient_diagnostic']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['test_date'])
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'patient_diagnostic', 'admission')
check_foreign_key(df, 'test_id', cleaned_dataframes['diagnostic_test'], 'test_id', 'patient_diagnostic', 'diagnostic_test')
check_foreign_key(df, 'doctor_id', cleaned_dataframes['doctor'], 'doctor_id', 'patient_diagnostic', 'doctor')
report_missing(df, 'patient_diagnostic'); report_duplicates(df, 'patient_diagnostic')
cleaned_dataframes['patient_diagnostic'] = df

# --- prescription ---
df = cleaned_dataframes['prescription']
df = strip_string_columns(df)
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'prescription', 'admission')
check_foreign_key(df, 'drug_id', cleaned_dataframes['drug'], 'drug_id', 'prescription', 'drug')
report_missing(df, 'prescription'); report_duplicates(df, 'prescription')
cleaned_dataframes['prescription'] = df

[diagnostic_test.department_id -> department.department_id] orphan rows: 0
[diagnostic_test] no missing values
[diagnostic_test] duplicate rows: 0
[patient_diagnostic.admission_id -> admission.admission_id] orphan rows: 0
[patient_diagnostic.test_id -> diagnostic_test.test_id] orphan rows: 0
[patient_diagnostic.doctor_id -> doctor.doctor_id] orphan rows: 0
[patient_diagnostic] no missing values
[patient_diagnostic] duplicate rows: 0
[prescription.admission_id -> admission.admission_id] orphan rows: 0
[prescription.drug_id -> drug.drug_id] orphan rows: 0
[prescription] no missing values
[prescription] duplicate rows: 0


### Financial Transactions

In [9]:
# --- billing ---
df = cleaned_dataframes['billing']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['bill_date'])

# Sanity check: insurance_covered + patient_payable should add up to total_amount.
mismatch = (df['insurance_covered_amount'] + df['patient_payable_amount'] - df['total_amount']).abs() > 1
print(f"[billing] rows where covered+payable != total: {mismatch.sum()}")

check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'billing', 'admission')
report_missing(df, 'billing'); report_duplicates(df, 'billing')
cleaned_dataframes['billing'] = df

# --- billing_detail ---
df = cleaned_dataframes['billing_detail']
df = strip_string_columns(df)
# reference_id is NaN for charge types like 'Room' that don't point to a
# specific drug/test - that's expected, not an error. Converted to
# nullable Int64 so it stays a whole number instead of an awkward float.
df['reference_id'] = df['reference_id'].astype('Int64')
check_foreign_key(df, 'bill_id', cleaned_dataframes['billing'], 'bill_id', 'billing_detail', 'billing')
report_missing(df, 'billing_detail')  # ~60% missing on reference_id is EXPECTED, not a cleaning failure
report_duplicates(df, 'billing_detail')
cleaned_dataframes['billing_detail'] = df

[billing] rows where covered+payable != total: 0
[billing.admission_id -> admission.admission_id] orphan rows: 0
[billing] no missing values
[billing] duplicate rows: 0
[billing_detail.bill_id -> billing.bill_id] orphan rows: 0
[billing_detail] missing values:
reference_id    59.97
dtype: float64

[billing_detail] duplicate rows: 0


[billing_detail] duplicate rows: 0


In [10]:
for table_name, df in cleaned_dataframes.items():
    df.to_csv(HMIS_PROCESSED_DIR / f"{table_name}.csv", index=False)
    print(f"Saved {table_name} -> {HMIS_PROCESSED_DIR / f'{table_name}.csv'}")

Saved department -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital HMIS Dataset for Healthcare Analytics\department.csv
Saved patient -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital HMIS Dataset for Healthcare Analytics\patient.csv
Saved employee -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital HMIS Dataset for Healthcare Analytics\employee.csv
Saved disease -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital HMIS Dataset for Healthcare Analytics\disease.csv
Saved insurance_provider -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Pat

Saved patient -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/patient.csv
Saved employee -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/employee.csv
Saved disease -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/disease.csv
Saved insurance_provider -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/insurance_provider.csv
Saved drug_manufacturer -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/drug_manufacturer.csv
Saved doctor -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/doctor.csv
Saved ward -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/ward.csv
Saved drug -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/drug.csv
Saved patient_insu

Saved admission -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/admission.csv
Saved diagnostic_test -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/diagnostic_test.csv


Saved patient_diagnostic -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/patient_diagnostic.csv
Saved prescription -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/prescription.csv


Saved billing -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/billing.csv
Saved billing_detail -> /home/claude/medtrack_fresh/data/processed/Hospital HMIS Dataset for Healthcare Analytics/billing_detail.csv


## Part 2 — Beds Management Cleaning (4 tables)

In [11]:
beds_file_names = ["patients.csv", "services_weekly.csv", "staff.csv", "staff_schedule.csv"]
beds_dataframes = {f[:-4]: pd.read_csv(BEDS_RAW_DIR / f) for f in beds_file_names}
beds_cleaned = {k: v.copy(deep=True) for k, v in beds_dataframes.items()}

# --- patients.csv ---
# Note: patient_id format (PAT-xxxxxxxx) confirms these are NOT the same
# patients as HMIS. Keep this table separate, never attempt a patient-level
# join with HMIS.
df = beds_cleaned['patients']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['arrival_date', 'departure_date'])
print(f"[patients] rows with departure before arrival: {(df['departure_date'] < df['arrival_date']).sum()}")
print(f"[patients] rows with implausible age: {((df['age'] < 0) | (df['age'] > 120)).sum()}")
print(f"[patients] duplicate patient_id count: {df['patient_id'].duplicated().sum()}")
report_missing(df, 'patients'); report_duplicates(df, 'patients')
beds_cleaned['patients'] = df

# --- staff.csv ---
df = beds_cleaned['staff']
df = strip_string_columns(df)
print(f"[staff] duplicate staff_id count: {df['staff_id'].duplicated().sum()}")
report_missing(df, 'staff'); report_duplicates(df, 'staff')
beds_cleaned['staff'] = df

# --- services_weekly.csv ---
df = beds_cleaned['services_weekly']
df = strip_string_columns(df)
df['event'] = df['event'].str.lower()
print(f"[services_weekly] rows where admitted > requested: {(df['patients_admitted'] > df['patients_request']).sum()}")
print(f"[services_weekly] rows where admitted > available_beds: {(df['patients_admitted'] > df['available_beds']).sum()}")
report_missing(df, 'services_weekly'); report_duplicates(df, 'services_weekly')
beds_cleaned['services_weekly'] = df

# --- staff_schedule.csv ---
# IMPORTANT: staff_id has 0% overlap with staff.csv - confirmed by a direct
# set check, each file assigns its own random ID independently. The real,
# working link between these two files is staff_name: all 110 names in
# staff.csv appear in staff_schedule.csv (which has 16 additional names
# staff.csv doesn't have). Checking staff_id here would report 100%
# orphans and look broken; checking staff_name gives the true picture.
df = beds_cleaned['staff_schedule']
df = strip_string_columns(df)
check_foreign_key(df, 'staff_name', beds_cleaned['staff'], 'staff_name', 'staff_schedule', 'staff')
print(f"[staff_schedule] duplicate (staff_id, week) rows: {df.duplicated(subset=['staff_id','week']).sum()}")
report_missing(df, 'staff_schedule'); report_duplicates(df, 'staff_schedule')
beds_cleaned['staff_schedule'] = df

[patients] rows with departure before arrival: 0
[patients] rows with implausible age: 0
[patients] duplicate patient_id count: 0
[patients] no missing values
[patients] duplicate rows: 0
[staff] duplicate staff_id count: 0
[staff] no missing values
[staff] duplicate rows: 0
[services_weekly] rows where admitted > requested: 0
[services_weekly] rows where admitted > available_beds: 0
[services_weekly] no missing values
[services_weekly] duplicate rows: 0
[staff_schedule.staff_name -> staff.staff_name] orphan rows: 832
[staff_schedule] duplicate (staff_id, week) rows: 0
[staff_schedule] no missing values
[staff_schedule] duplicate rows: 0


In [12]:
for table_name, df in beds_cleaned.items():
    df.to_csv(BEDS_PROCESSED_DIR / f"beds_{table_name}.csv", index=False)
    print(f"Saved {table_name} -> {BEDS_PROCESSED_DIR / f'beds_{table_name}.csv'}")

Saved patients -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital Beds Management\beds_patients.csv
Saved services_weekly -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital Beds Management\beds_services_weekly.csv
Saved staff -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital Beds Management\beds_staff.csv
Saved staff_schedule -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital Beds Management\beds_staff_schedule.csv


## Part 3 — Readmission Dataset Cleaning (1 table)

In [13]:
df = pd.read_csv(READMISSION_RAW_DIR / "Healthcare Data Analysis for readmission.csv")
cleaned_df = df.copy(deep=True)
cleaned_df = strip_string_columns(cleaned_df)

# Rename ID columns NOW - unrelated ID space to HMIS despite similar names,
# renaming up front means they can never accidentally get treated as HMIS's
# own patient_id/doctor_id/hospital_id downstream.
cleaned_df = cleaned_df.rename(columns={
    'patient_id': 'readm_patient_id', 'doctor_id': 'readm_doctor_id', 'hospital_id': 'readm_hospital_id',
})

# Mixed date formats seen in profiling (DD-MM-YYYY and DD/MM/YYYY both
# appear in the same column) - format='mixed' handles both.
date_cols = ['Admission_date', 'patient_checkin_date', 'patient_checkout_date']
for col in date_cols:
    cleaned_df[col] = pd.to_datetime(cleaned_df[col], format='mixed', dayfirst=True, errors='coerce')

# FLAG (do not fix) the operational fields that fail internal logic checks.
# We do NOT overwrite, drop, or "correct" these - that would mean inventing
# data we don't have. We only mark them so nothing downstream treats them
# as verified facts.
cleaned_df['dates_reliable'] = ~(cleaned_df['patient_checkout_date'] < cleaned_df['patient_checkin_date'])
cleaned_df['bed_counts_reliable'] = ~(cleaned_df['occupied_beds'] > cleaned_df['hospital_beds_available'])
cleaned_df['length_of_stay_reliable'] = (
    (cleaned_df['patient_checkout_date'] - cleaned_df['patient_checkin_date']).dt.days
    == cleaned_df['patient_length_of_stay']
)
print(f"rows with unreliable dates: {(~cleaned_df['dates_reliable']).sum()} / {len(cleaned_df)}")
print(f"rows with unreliable bed counts: {(~cleaned_df['bed_counts_reliable']).sum()} / {len(cleaned_df)}")
print(f"rows with unreliable length_of_stay: {(~cleaned_df['length_of_stay_reliable']).sum()} / {len(cleaned_df)}")
print("NOTE: readm_hospital_id / readm_doctor_id / time_slot NOT treated as reliable identifiers")
print(f"patient_sat_score range: {cleaned_df['patient_sat_score'].min()}-{cleaned_df['patient_sat_score'].max()} "
      f"- scale/meaning undocumented by the source, NOT assumed to be 0-100. Left as raw value.")

report_missing(cleaned_df, 'readmission'); report_duplicates(cleaned_df, 'readmission')

cleaned_df.to_csv(READMISSION_PROCESSED_DIR / "readmission_dataset.csv", index=False)
print(f"Saved -> {READMISSION_PROCESSED_DIR / 'readmission_dataset.csv'}")

rows with unreliable dates: 4750 / 10000
rows with unreliable bed counts: 2777 / 10000
rows with unreliable length_of_stay: 9822 / 10000
NOTE: readm_hospital_id / readm_doctor_id / time_slot NOT treated as reliable identifiers
patient_sat_score range: 100-1600 - scale/meaning undocumented by the source, NOT assumed to be 0-100. Left as raw value.
[readmission] no missing values
[readmission] duplicate rows: 0
Saved -> C:\Users\sr189\OneDrive\Desktop\Hospital-Performance-Intelligence-System-for-Operational-and-Patient-Care-Analytics\m1 shivam\data\processed\Hospital Data for Patient Readmission Prediction\readmission_dataset.csv


Saved -> /home/claude/medtrack_fresh/data/processed/Hospital Data for Patient Readmission Prediction/readmission_dataset.csv
